In [7]:
#Intalacion para funcionamiento del bot de Eduardo
!pip install -q discord.py nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.2 MB/s eta 0:00:00


In [8]:
#Celda para subir archivos

'''from google.colab import drive, files
drive.mount('/content/drive')

import os, sys, importlib
CARPETA_CODIGO = "/content/drive/MyDrive/Colab Notebooks/Codigo unificado con imports"
os.chdir(CARPETA_CODIGO)
sys.path.insert(0, CARPETA_CODIGO)

# archivos que vas a reemplazar: se guardan antes como .bak para que el nuevo
# conserve su nombre exacto (si no, Colab lo sube como "nombre (1).py")
REEMPLAZAR = ["armar_json_bot_python.py"]
for nombre in REEMPLAZAR:
    if os.path.exists(nombre):
        os.replace(nombre, nombre + ".bak")
        print("respaldo:", nombre, "->", nombre + ".bak")

files.upload()
importlib.invalidate_caches()
print(sorted(os.listdir()))
'''

'from google.colab import drive, files\ndrive.mount(\'/content/drive\')\n\nimport os, sys, importlib\nCARPETA_CODIGO = "/content/drive/MyDrive/Colab Notebooks/Codigo unificado con imports"\nos.chdir(CARPETA_CODIGO)\nsys.path.insert(0, CARPETA_CODIGO)\n\n# archivos que vas a reemplazar: se guardan antes como .bak para que el nuevo\n# conserve su nombre exacto (si no, Colab lo sube como "nombre (1).py")\nREEMPLAZAR = ["armar_json_bot_python.py"]\nfor nombre in REEMPLAZAR:\n    if os.path.exists(nombre):\n        os.replace(nombre, nombre + ".bak")\n        print("respaldo:", nombre, "->", nombre + ".bak")\n\nfiles.upload()\nimportlib.invalidate_caches()\nprint(sorted(os.listdir()))\n'

In [9]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, importlib
from datetime import datetime, timezone
from types import SimpleNamespace as NS

CARPETA_CODIGO = "/content/drive/MyDrive/Colab Notebooks/Codigo unificado con imports"
os.chdir(CARPETA_CODIGO)
sys.path.insert(0, CARPETA_CODIGO)

import armar_json_bot_python, bot_lote
importlib.reload(armar_json_bot_python)  # primero este, porque bot_lote lo importa
importlib.reload(bot_lote)               # cliente nuevo en cada corrida (el anterior se cierra al terminar)

# verificación por comportamiento: los mensajes deben salir de más viejo a más nuevo
viejo = NS(author="a", channel=NS(id=1), content="viejo", created_at=datetime(2026, 1, 1, tzinfo=timezone.utc))
nuevo = NS(author="b", channel=NS(id=1), content="nuevo", created_at=datetime(2026, 1, 2, tzinfo=timezone.utc))
orden = [i["texto"] for i in bot_lote.construir_paquete([nuevo, viejo])["interacciones"]]
assert orden == ["viejo", "nuevo"], f"Sigue la versión vieja (salió {orden})"

from base_datos import crear_tabla
crear_tabla()  # si la tabla ya existe, no hace nada

bot_lote.cliente.run(bot_lote.TOKEN)

Mounted at /content/drive


2026-10-01 01:43:50 INFO     discord.client logging in using static token
INFO:discord.client:logging in using static token
2026-10-01 01:43:51 INFO     discord.gateway Shard ID None has connected to Gateway (Session ID: 46ae88608e09fcf49774521d3614e4d1).
INFO:discord.gateway:Shard ID None has connected to Gateway (Session ID: 46ae88608e09fcf49774521d3614e4d1).


Conectado como CommunityLab Bot#7078
Arme el paquete con 40 interacciones, lo proceso...
{'status': 'exito', 'periodo_referencia': 'Semana_03', 'total_recibidas': 40, 'guardadas_nuevas': 0, 'repetidas_omitidas': 40, 'vacias_omitidas': 0}


In [10]:
#from google.colab import drive
#drive.mount('/content/drive', force_remount=True)

import os, sys, importlib, json
CARPETA_CODIGO = "/content/drive/MyDrive/Colab Notebooks/Codigo unificado con imports"
os.chdir(CARPETA_CODIGO)
sys.path.insert(0, CARPETA_CODIGO)
importlib.invalidate_caches()
print(sorted(os.listdir()))

from ver_json import obtener_json
print(json.dumps(obtener_json(), indent=2, ensure_ascii=False))

['Codigo_con_imports.ipynb', '__pycache__', 'armar_json_bot_python.py', 'base_datos.py', 'bot_lote.py', 'cerebro_ia.py', 'cerebro_ia_gemini_3.5_flash.py', 'cerebro_ia_gemma_4.py', 'communitylab.db', 'limpieza.py', 'modelos.py', 'procesamiento.py', 'resultados_kpis.json', 'ver_json.py']
{
  "total": 40,
  "interacciones": [
    {
      "id": 40,
      "origen_comunidad": "Discord_Grupo_ONE_G10",
      "periodo_referencia": "Semana_03",
      "autor": "dosguel",
      "canal": "#preguntas",
      "tipo": null,
      "texto": "Me encanta que el bot de ayuda responda “consulta el README” justo cuando el README es lo que no funciona. Diez puntos para la automatización.",
      "estado": "pendiente",
      "creado_en": "2026-09-30T05:12:26"
    },
    {
      "id": 39,
      "origen_comunidad": "Discord_Grupo_ONE_G10",
      "periodo_referencia": "Semana_03",
      "autor": "dosguel",
      "canal": "#preguntas",
      "tipo": null,
      "texto": "Estoy probando una integración. Mi correo f

In [18]:
# Curaduría
# 1. Instalación de las librerías (Agregamos la de Google)
!pip install -q langchain-groq langchain-core pydantic langchain-google-genai

import os, sys, importlib, json
from dotenv import load_dotenv
load_dotenv()  # Carga las variables de entorno desde el archivo .env

# 2. Configuración de las llaves en el entorno de Colab
os.environ["GROQ_API_KEY"] = os.environ.get('GROQ_API_KEY')
os.environ["GOOGLE_API_KEY"] = os.environ.get('GOOGLE_API_KEY') # Asegúrate de tenerla en los secretos de Colab
os.environ["GROQ_MODEL"] = "openai/gpt-oss-120b"

# ====================================================================
# 3. SELECTOR DE MOTOR IA (Descomenta SOLO el que quieras usar)
# ====================================================================

# 🔴 OPCIÓN 1: Groq (openai/gpt-oss-120b) - (Rapido pero limitado en cuanto a peticiones por dia, 1K peticiones por dia - peticiones por minuto 30 - 8k por tokens por minuto )
import cerebro_ia

# 🔵 OPCIÓN 2: Google Gemma 4 31B (Tardado pero con mayor libertad de peticiones, 14.4K peticiones por dia - peticiones por minuto 15 - 16k por tokens por minuto )
#import cerebro_ia_gemma_4 as cerebro_ia

# 🟢 OPCIÓN 3: Google Gemini 3.5 Flash Lite (Rapido con 500 petiones al dia - peticiones por minuto 15 - 250k tokens por minuto)
#import cerebro_ia_gemini_flash as cerebro_ia

# ====================================================================

from ver_json import obtener_json

# Recarga por si se hacen cambios recientes en tu archivo en Drive
importlib.reload(cerebro_ia)

# 4. INGESTA: Se toman los datos de Eduardo
paquete_lalo = obtener_json()
mensajes = paquete_lalo.get("interacciones", [])

resultados_destacados = []
resultados_descartados = []

print(f"🧠 Módulo Cerebro importado con éxito. Analizando {len(mensajes)} mensajes...\n")

# Se analizan solo los primeros 4 para no quemar tokens en pruebas
for msg in mensajes[:4]:
    print(f"Analizando a {msg['autor']}...")

    # Llamada a la cadena (Funciona idéntico sin importar el motor que elegiste arriba)
    respuesta = cerebro_ia.cadena.invoke({"texto": msg["texto"], "canal": msg["canal"]})
    kpis = respuesta.model_dump()

    dato = {
        "id_mensaje": msg["id"],
        "autor": msg["autor"],
        "canal_origen": msg["canal"],
        "texto_original": msg["texto"],
        "kpis": kpis
    }

    if kpis["relevancia"] >= 70:
        resultados_destacados.append(dato)
    else:
        resultados_descartados.append(dato)

print("\n✅ ANÁLISIS COMPLETADO")
print(f"🔥 Aprobados: {len(resultados_destacados)}")
print(f"🗑️ Descartados: {len(resultados_descartados)}")

# 5. Empaquetado y guardado persistente en Drive
paquete_maestro = {
    "aprobados": resultados_destacados,
    "descartados": resultados_descartados
}

with open("resultados_kpis.json", "w", encoding="utf-8") as archivo:
    json.dump(paquete_maestro, archivo, indent=2, ensure_ascii=False)

print("💾 Archivo 'resultados_kpis.json' guardado en Drive para el equipo.")
print("\n--- JSON FINAL PARA COPYWRITING ---")
print(json.dumps(paquete_maestro, indent=2, ensure_ascii=False))

🧠 Módulo Cerebro importado con éxito. Analizando 40 mensajes...

Analizando a dosguel...
Analizando a dosguel...
Analizando a dosguel...
Analizando a dosguel...

✅ ANÁLISIS COMPLETADO
🔥 Aprobados: 2
🗑️ Descartados: 2
💾 Archivo 'resultados_kpis.json' guardado en Drive para el equipo.

--- JSON FINAL PARA COPYWRITING ---
{
  "aprobados": [
    {
      "id_mensaje": 38,
      "autor": "dosguel",
      "canal_origen": "#general",
      "texto_original": "La sesión de agentes fue interesante, pero faltó un ejemplo que muestre qué ocurre cuando el modelo devuelve JSON incompleto. Tres compañeros tuvimos el mismo error y cada uno lo resolvió de manera distinta. Sería útil agregar una prueba automatizada y explicar cuál es el comportamiento esperado.",
      "kpis": {
        "relevancia": 85,
        "sentimiento": "Neutral",
        "temas_clave": "#agentes, #JSONincompleto, #pruebasautomatizadas",
        "segmento": "Desarrolladores",
        "razonamiento": "El mensaje brinda feedback con